In [1]:
import pandas as pd
import warnings

warnings.filterwarnings('ignore')

### Objectives:-
1. Read car sales data as csv
2. Perform data cleaning by checking for nulls and dropping unwanted columns 
3. Perform feature encoding using LabelEncoder, OneHotEncoder, and StandardScaler
4. Train, test, split data
5. Train multiple Regression models (GradientBoostingRegressor, AdaBoostRegressor, LinearRegression, Ridge, Lasso, RandomForestRegressor, KNeighborsRegressor, DecisionTreeRegressor)
6. Print and review performance metrics (r2_score, mean_absolute_error, mean_squared_error) for both test and training data
7. Take two models (GradientBoostingRegressor and RandomForestRegressor) and fine tune them by passing hyperparameters using RandomizedSearchCV
8. Find best params
9. Rerun GradientBoostingRegressor and RandomForestRegressor models with the best parameters
10. Print and review performance metrics (r2_score, mean_absolute_error, mean_squared_error) for both test and training data


In [2]:
df = pd.read_csv('../data/raw/cardekho_imputated.csv')
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 15411 entries, 0 to 15410
Data columns (total 14 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Unnamed: 0         15411 non-null  int64  
 1   car_name           15411 non-null  str    
 2   brand              15411 non-null  str    
 3   model              15411 non-null  str    
 4   vehicle_age        15411 non-null  int64  
 5   km_driven          15411 non-null  int64  
 6   seller_type        15411 non-null  str    
 7   fuel_type          15411 non-null  str    
 8   transmission_type  15411 non-null  str    
 9   mileage            15411 non-null  float64
 10  engine             15411 non-null  int64  
 11  max_power          15411 non-null  float64
 12  seats              15411 non-null  int64  
 13  selling_price      15411 non-null  int64  
dtypes: float64(2), int64(6), str(6)
memory usage: 1.6 MB


### Data Cleaning

In [3]:
df.drop('Unnamed: 0', axis=1, inplace=True)
df.head()

,car_name,brand,model,vehicle_age,km_driven,seller_type,fuel_type,transmission_type,mileage,engine,max_power,seats,selling_price
0,Maruti Alto,Maruti,Alto,9,120000,Individual,Petrol,Manual,19.70,796,46.30,5,120000
1,Hyundai Grand,Hyundai,Grand,5,20000,Individual,Petrol,Manual,18.90,1197,82.00,5,550000
2,Hyundai i20,Hyundai,i20,11,60000,Individual,Petrol,Manual,17.00,1197,80.00,5,215000
3,Maruti Alto,Maruti,Alto,9,37000,Individual,Petrol,Manual,20.92,998,67.10,5,226000
4,Ford Ecosport,Ford,Ecosport,6,30000,Dealer,Diesel,Manual,22.77,1498,98.59,5,570000


In [4]:
df.isnull().sum()

car_name             0
brand                0
model                0
vehicle_age          0
km_driven            0
seller_type          0
fuel_type            0
transmission_type    0
mileage              0
engine               0
max_power            0
seats                0
selling_price        0
dtype: int64

In [5]:
# out 0f drop car_name and brand and model let's drop 2 columns as it's repetitive infomration

df.drop(['brand', 'car_name'], axis=1, inplace=True)
df.head()

,model,vehicle_age,km_driven,seller_type,fuel_type,transmission_type,mileage,engine,max_power,seats,selling_price
0,Alto,9,120000,Individual,Petrol,Manual,19.70,796,46.30,5,120000
1,Grand,5,20000,Individual,Petrol,Manual,18.90,1197,82.00,5,550000
2,i20,11,60000,Individual,Petrol,Manual,17.00,1197,80.00,5,215000
3,Alto,9,37000,Individual,Petrol,Manual,20.92,998,67.10,5,226000
4,Ecosport,6,30000,Dealer,Diesel,Manual,22.77,1498,98.59,5,570000


In [6]:
# split data into dependent and independent features
X, y = df.drop('selling_price', axis=1), df['selling_price']

### Feature Encoding 

In [7]:
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler

label_encoder = LabelEncoder()
X['model'] = label_encoder.fit_transform(X['model'])

In [8]:
# Create Column transformer
from sklearn.compose import ColumnTransformer

num_feat = X.select_dtypes(exclude='str').columns
one_hot_columns = ['seller_type', 'fuel_type', 'transmission_type']

num_transformer = StandardScaler()
one_hot_transformer = OneHotEncoder(drop='first')

preprocessor = ColumnTransformer([("text_preprocess", one_hot_transformer, one_hot_columns),("num_preprocess", num_transformer, num_feat)],
                                  remainder='passthrough')

X = preprocessor.fit_transform(X)

In [9]:
# train, test, split
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

In [12]:
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, AdaBoostRegressor, GradientBoostingRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.model_selection import RandomizedSearchCV

In [11]:
def evaluate_model(true_val, predicted_value):
    mse = mean_squared_error(true_val, predicted_value)
    mae = mean_absolute_error(true_val, predicted_value)
    r2 = r2_score(true_val, predicted_value)

    return mse, mae, r2


In [ ]:
# Let's test multiple linear regression model we have learned so far including Gradient Boosting Regressor

models = {'Random Forest Regressor': RandomForestRegressor(), 'Linear Regression': LinearRegression(), 'Ridge': Ridge(), 'Lasso': Lasso(),
          'K Neighbors Regressor': KNeighborsRegressor(), 'Decision Tree Regressor': DecisionTreeRegressor(),
          'AdaBoost Regressor': AdaBoostRegressor(),
          'Gradient Boosting Regressor': GradientBoostingRegressor()}

for k, v in models.items():
    model = v
    model.fit(X_train, y_train)

    # make predictions
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    mse_train, mae_train, r2_train = evaluate_model(y_train, y_train_pred)
    mse_test, mae_test, r2_test = evaluate_model(y_test, y_test_pred)

    print('===========================')
    print(k,':- Metrics')
    print('===========================')
    print('Train Model Performance Metrics ')
    print(f'root_mean_squared_error:- {mse_train}')
    print(f'mean_absolute_error:- {mae_train}')
    print(f'r2_score:- {r2_train}')

    print('-------------------')

    print('Test Model Performance Metrics ')
    print(f'root_mean_squared_error:- {mse_test}')
    print(f'mean_absolute_error:- {mae_test}')
    print(f'r2_score:- {r2_test}')

    print('\n')




Random Forest Regressor :- Metrics
Train Model Performance Metrics 
root_mean_squared_error:- 17401683638.377922
mean_absolute_error:- 39743.149903536236
r2_score:- 0.978543850474654
-------------------
Test Model Performance Metrics 
root_mean_squared_error:- 52726013924.72612
mean_absolute_error:- 102264.48504252802
r2_score:- 0.9299584360417626


Linear Regression :- Metrics
Train Model Performance Metrics 
root_mean_squared_error:- 306756099359.7596
mean_absolute_error:- 268101.6070829937
r2_score:- 0.6217719576765959
-------------------
Test Model Performance Metrics 
root_mean_squared_error:- 252550062888.5656
mean_absolute_error:- 279618.57941584283
r2_score:- 0.6645109298852004


Ridge :- Metrics
Train Model Performance Metrics 
root_mean_squared_error:- 306756818740.9266
mean_absolute_error:- 268059.8014688303
r2_score:- 0.6217710706848424
-------------------
Test Model Performance Metrics 
root_mean_squared_error:- 252540243247.9684
mean_absolute_error:- 279557.2168930266
r2_

In [15]:
# Let's fine tune Gradient Boost and Randforest models with some parameters and see if we can improve performance

params_gradient = {'loss':['squared_error', 'absolute_error', 'huber'],
                   'criterion': ['friedman_mse', 'squared_error'],
                   'min_samples_split': [2, 8, 15, 20],
                   'n_estimators': [100, 200, 500, 1000],
                   'max_depth': [None, 5, 8, 10, 15],
                   'learning_rate': [0.1, 0.01, 0.02, 0.03]
                   }
rf_params = {'max_depth': [None, 5,8,10, 15], 'max_features': [5,7,8, 'auto'], 'min_samples_split': [2,8,15,20], 'n_estimators': [100, 200, 500, 1000]}


In [16]:
random_cv_model = [('GBR', GradientBoostingRegressor(), params_gradient), 
                   ('RF', RandomForestRegressor(), rf_params),]

In [17]:
# Initialize and fit models with hyperparameters
model_best_param = {}

for name, model, params in random_cv_model:
    random = RandomizedSearchCV(estimator=model, param_distributions=params, n_iter=100, cv=3, verbose=2, n_jobs=-1)
    random.fit(X_train, y_train)
    model_best_param[name] = random.best_params_

for k, v in model_best_param.items():
    print(f'--------Best Params For Model {k} ------------')
    print(v)

Fitting 3 folds for each of 100 candidates, totalling 300 fits
[CV] END criterion=friedman_mse, learning_rate=0.01, loss=huber, max_depth=8, min_samples_split=15, n_estimators=100; total time=  15.5s
[CV] END criterion=friedman_mse, learning_rate=0.01, loss=huber, max_depth=8, min_samples_split=15, n_estimators=100; total time=  15.9s
[CV] END criterion=friedman_mse, learning_rate=0.01, loss=huber, max_depth=8, min_samples_split=15, n_estimators=100; total time=  16.0s
[CV] END criterion=friedman_mse, learning_rate=0.1, loss=squared_error, max_depth=15, min_samples_split=20, n_estimators=500; total time=  34.3s
[CV] END criterion=friedman_mse, learning_rate=0.1, loss=squared_error, max_depth=15, min_samples_split=20, n_estimators=500; total time=  34.5s
[CV] END criterion=friedman_mse, learning_rate=0.1, loss=squared_error, max_depth=15, min_samples_split=20, n_estimators=500; total time=  34.5s
[CV] END criterion=squared_error, learning_rate=0.02, loss=absolute_error, max_depth=10, mi

In [19]:
# let's retrain our models with best params

models = {'Random Forest Regressor': RandomForestRegressor(n_estimators=model_best_param['RF']['n_estimators'],
                                                              min_samples_split=model_best_param['RF']['min_samples_split'], 
                                                              max_features=model_best_param['RF']['max_features'], 
                                                              max_depth=model_best_param['RF']['max_depth'], n_jobs=-1),
                                                              'Gradient Boost REgressor': GradientBoostingRegressor(n_estimators=model_best_param['GBR']['n_estimators'],
                                                              min_samples_split=model_best_param['GBR']['min_samples_split'],
                                                              max_depth=model_best_param['GBR']['max_depth'],
                                                                loss=model_best_param['GBR']['loss'],
                                                                learning_rate=model_best_param['GBR']['learning_rate'],
                                                                criterion=model_best_param['GBR']['criterion'])}

for k, v in models.items():
    model = v
    model.fit(X_train, y_train)

    # make predictions
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    mse_train, mae_train, r2_train = evaluate_model(y_train, y_train_pred)
    mse_test, mae_test, r2_test = evaluate_model(y_test, y_test_pred)

    print('===========================')
    print(k,':- Metrics')
    print('===========================')
    print('Train Model Performance Metrics ')
    print(f'root_mean_squared_error:- {mse_train}')
    print(f'mean_absolute_error:- {mae_train}')
    print(f'r2_score:- {r2_train}')

    print('-------------------')

    print('Test Model Performance Metrics ')
    print(f'root_mean_squared_error:- {mse_test}')
    print(f'mean_absolute_error:- {mae_test}')
    print(f'r2_score:- {r2_test}')

    print('\n')


Random Forest Regressor :- Metrics
Train Model Performance Metrics 
root_mean_squared_error:- 16246106210.332369
mean_absolute_error:- 39524.35729150257
r2_score:- 0.9799686690496555
-------------------
Test Model Performance Metrics 
root_mean_squared_error:- 44036235492.503395
mean_absolute_error:- 98294.40982392208
r2_score:- 0.9415019916140153


Gradient Boost REgressor :- Metrics
Train Model Performance Metrics 
root_mean_squared_error:- 8835100329.120903
mean_absolute_error:- 54970.54757759754
r2_score:- 0.9891063854697957
-------------------
Test Model Performance Metrics 
root_mean_squared_error:- 90372410100.74677
mean_absolute_error:- 100221.53171901505
r2_score:- 0.8799487298401082


